In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sms-spam-collection-dataset/spam.csv


In [2]:
data = pd.read_csv('/kaggle/input/sms-spam-collection-dataset/spam.csv',encoding='ISO-8859-1')

In [3]:
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


### Text Cleaning

In [4]:
data = data[['v1','v2']]

In [5]:
data = data.applymap(lambda x: x.lower() if isinstance(x,str) else x)

/tmp/ipykernel_35/238800201.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(lambda x: x.lower() if isinstance(x,str) else x)


In [6]:
data

,v1,v2
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,this is the 2nd time we have tried 2 contact u...
5568,ham,will ì_ b going to esplanade fr home?
5569,ham,"pity, * was in mood for that. so...any other s..."
5570,ham,the guy did some bitching but i acted like i'd...


In [7]:
data.isnull().sum()

v1    0
v2    0
dtype: int64

#### Removing URLs

In [8]:
data['v2'] = data['v2'].astype(str)

In [9]:
data

,v1,v2
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,this is the 2nd time we have tried 2 contact u...
5568,ham,will ì_ b going to esplanade fr home?
5569,ham,"pity, * was in mood for that. so...any other s..."
5570,ham,the guy did some bitching but i acted like i'd...


In [10]:
import re

# Defining regex pattern to match URLs
url_pattern = re.compile(f'https?://\S+')

def remove_urls(text):
    return url_pattern.sub("",text)

In [11]:
data['v2'] = data['v2'].apply(remove_urls)

Removing non-word and non-whitespace characters

In [12]:
data = data.replace(to_replace=r'[^\w\s]',value='',regex=True)

In [13]:
data

,v1,v2
0,ham,go until jurong point crazy available only in ...
1,ham,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor u c already then say
4,ham,nah i dont think he goes to usf he lives aroun...
...,...,...
5567,spam,this is the 2nd time we have tried 2 contact u...
5568,ham,will ì_ b going to esplanade fr home
5569,ham,pity was in mood for that soany other suggest...
5570,ham,the guy did some bitching but i acted like id ...


#### Tokenization

- Tokenization is the process of breaking down large blocks of text such as paragraphs and sentences into smaller, more manageable units.

  ![image.png](attachment:3897f0b8-a8ed-48ab-874a-b780d6d5e899.png)

In [14]:
import nltk
from nltk.tokenize import word_tokenize

data['v2'] = data['v2'].apply(word_tokenize)

In [15]:
data

,v1,v2
0,ham,"[go, until, jurong, point, crazy, available, o..."
1,ham,"[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,ham,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"[nah, i, dont, think, he, goes, to, usf, he, l..."
...,...,...
5567,spam,"[this, is, the, 2nd, time, we, have, tried, 2,..."
5568,ham,"[will, ì_, b, going, to, esplanade, fr, home]"
5569,ham,"[pity, was, in, mood, for, that, soany, other,..."
5570,ham,"[the, guy, did, some, bitching, but, i, acted,..."


### Stopword Removal

- Stopwords refer to the most commonly occurring words in any natural language.

  ![image.png](attachment:e23edcac-9bae-4894-87b6-396ab460169f.png)

In [16]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [17]:
for idx,i in enumerate(stop_words):
    print(i)
    if idx == 20:
        break

mightn't
aren
i
their
yourself
won't
only
under
both
didn
y
isn't
hadn
himself
above
it
before
out
his
yourselves
him


In [18]:
data['v2'] = data['v2'].apply(lambda x: [word for word in x if word not in stop_words])

In [19]:
data

,v1,v2
0,ham,"[go, jurong, point, crazy, available, bugis, n..."
1,ham,"[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,ham,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,"[nah, dont, think, goes, usf, lives, around, t..."
...,...,...
5567,spam,"[2nd, time, tried, 2, contact, u, u, å750, pou..."
5568,ham,"[ì_, b, going, esplanade, fr, home]"
5569,ham,"[pity, mood, soany, suggestions]"
5570,ham,"[guy, bitching, acted, like, id, interested, b..."


#### Stemming
Stemming is a rule-based heuristic process that chops off the ends of words (suffixes) to get to a "stem." It's a faster and simpler method, but it doesn't always result in a linguistically correct or meaningful word. It's like a blunt instrument that just cuts off the end.

Example:

"running" -> "run"
"jumps" -> "jump"
"studies" -> "studi" (This isn't a real word, but it's the stem)
"universal" -> "univers" (Again, not a real word)


In [20]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_words(words):
    return [stemmer.stem(word) for word in words]

data['stemmed_v2'] = data['v2'].apply(stem_words)

In [21]:
data

,v1,v2,stemmed_v2
0,ham,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,"[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho..."
...,...,...,...
5567,spam,"[2nd, time, tried, 2, contact, u, u, å750, pou...","[2nd, time, tri, 2, contact, u, u, å750, pound..."
5568,ham,"[ì_, b, going, esplanade, fr, home]","[ì_, b, go, esplanad, fr, home]"
5569,ham,"[pity, mood, soany, suggestions]","[piti, mood, soani, suggest]"
5570,ham,"[guy, bitching, acted, like, id, interested, b...","[guy, bitch, act, like, id, interest, buy, som..."


#### Lemmatization

Lemmatization is a more sophisticated and linguistically informed process. It aims to return the base or dictionary form of a word, known as a "lemma." It does this by considering the word's morphological analysis and often uses a dictionary or a lexical database (like WordNet) to ensure the resulting lemma is a valid word. It also often takes into account the part-of-speech (POS) of the word to determine the correct lemma.

Example:

"running" -> "run"
"ran" -> "run"
"better" -> "good" (Here, it correctly identifies the root as "good" because it understands the meaning)
"geese" -> "goose"
"mice" -> "mouse"
"meeting" (noun) -> "meeting"
"meeting" (verb, as in "We are meeting") -> "meet"


In [22]:
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [23]:
# nltk.download('omw-1.4')
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /usr/share/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /usr/share/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /usr/share/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /usr/share/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /usr/share/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron

True

In [24]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    def get_wordnet_pos(word):
        tag = nltk.pos_tag([word])[0][1][0].upper()
        tag_dict = {
            "J": wordnet.ADJ,
            "N": wordnet.NOUN,
            "V": wordnet.VERB,
            "R": wordnet.ADV
        }
        return tag_dict.get(tag,wordnet.NOUN)

    lemmas = [lemmatizer.lemmatize(token,get_wordnet_pos(token)) for token in tokens]
    return lemmas


In [25]:
data['lemmatized_v2'] = data['v2'].apply(lemmatize_tokens)


In [26]:
data

,v1,v2,stemmed_v2,lemmatized_v2
0,ham,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazy, available, bugis, n..."
1,ham,"[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,ham,"[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, early, hor, u, c, already, say]"
4,ham,"[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, go, usf, life, around, though]"
...,...,...,...,...
5567,spam,"[2nd, time, tried, 2, contact, u, u, å750, pou...","[2nd, time, tri, 2, contact, u, u, å750, pound...","[2nd, time, try, 2, contact, u, u, å750, pound..."
5568,ham,"[ì_, b, going, esplanade, fr, home]","[ì_, b, go, esplanad, fr, home]","[ì_, b, go, esplanade, fr, home]"
5569,ham,"[pity, mood, soany, suggestions]","[piti, mood, soani, suggest]","[pity, mood, soany, suggestion]"
5570,ham,"[guy, bitching, acted, like, id, interested, b...","[guy, bitch, act, like, id, interest, buy, som...","[guy, bitching, act, like, id, interested, buy..."


In [32]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
sequence = """
Question: "Where is tokenization used ?"
Answer: "Tokenization is the first step in the processing stage."
"""
tokens = tokenizer.tokenize(sequence)

In [33]:
print(tokens)

['Question', ':', '"', 'Where', 'is', 'token', '##ization', 'used', '?', '"', 'Answer', ':', '"', 'To', '##ken', '##ization', 'is', 'the', 'first', 'step', 'in', 'the', 'processing', 'stage', '.', '"']


In [34]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[22171, 131, 107, 2777, 1110, 22559, 2734, 1215, 136, 107, 26018, 131, 107, 1706, 6378, 2734, 1110, 1103, 1148, 2585, 1107, 1103, 6165, 2016, 119, 107]


In [35]:
decoded_string = tokenizer.decode(ids)
print(decoded_string)

Question : " Where is tokenization used? " Answer : " Tokenization is the first step in the processing stage. "


In [31]:
encoded = tokenizer(sequence)
print(encoded)

{'input_ids': [101, 1706, 6378, 2734, 1110, 1103, 1148, 2585, 1107, 1103, 6165, 2016, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [1]:
from sklearn.preprocessing import OneHotEncoder
import itertools

document = ["The","boy","sat","on","the","floor"]

tokens = [doc.split(" ") for doc in document]
token_chain = itertools.chain.from_iterable(tokens)
word_to_id = {token: idx for idx, token in enumerate(set(token_chain))}
#word_to_id is our required dictionary
#Get the corresponding values for each word
token_ids = [[word_to_id[token] for token in toke] for toke in tokens]

vec = OneHotEncoder(categories="auto")
V = vec.fit_transform(token_ids)
print(V.toarray())


[[0. 0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]]


In [3]:
word_to_id

{'on': 0, 'boy': 1, 'sat': 2, 'the': 3, 'floor': 4, 'The': 5}

In [4]:
tokens

[['The'], ['boy'], ['sat'], ['on'], ['the'], ['floor']]

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
document = ["NLP has changed the world. I love NLP. NLP is cool"]
vectorizer = CountVectorizer()
# tokenize and build vocabulary
vectorizer.fit(document)
print(vectorizer.vocabulary_)
# encode document
vector = vectorizer.transform(text)
# summarize encoded vector
print(vector.toarray())

{'nlp': 5, 'has': 2, 'changed': 0, 'the': 6, 'world': 7, 'love': 4, 'is': 3, 'cool': 1}


ValueError: Iterable over raw text documents expected, string object received.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
text = ['NLP has changed the world', 'I love NLP', 'NLP is cool', 'NLP is future']
tf = TfidfVectorizer()
txt_fit = tf.fit(text)
txt_transform = txt_fit.transform(text)
idf = tf.idf_
print(dict(zip(txt_fit.get_feature_names_out(), idf)))


{'changed': 1.916290731874155, 'cool': 1.916290731874155, 'future': 1.916290731874155, 'has': 1.916290731874155, 'is': 1.5108256237659907, 'love': 1.916290731874155, 'nlp': 1.0, 'the': 1.916290731874155, 'world': 1.916290731874155}


In [10]:
text = ['NLP has changed the world', 'I love NLP', 'NLP is cool', 'NLP is future']
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(2,2))
bow = cv.fit_transform(text)
print(cv.vocabulary_)
print(bow[0].toarray())


{'nlp has': 5, 'has changed': 1, 'changed the': 0, 'the world': 7, 'love nlp': 4, 'nlp is': 6, 'is cool': 2, 'is future': 3}
[[1 1 0 0 0 1 0 1]]


In [11]:
from gensim.models import Word2Vec

# Sample data
sentences = [
    ['this', 'is', 'the', 'first', 'document'],
    ['this', 'document', 'is', 'the', 'second', 'document'],
    ['and', 'this', 'is', 'the', 'third', 'one'],
    ['is', 'this', 'the', 'first', 'document']
]

# Initialize the Word2Vec model
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Train the model
model.train(sentences, total_examples=len(sentences), epochs=10)

# Get vector for a word
print(model.wv['document'])

[-5.3622725e-04  2.3643136e-04  5.1033497e-03  9.0092728e-03
 -9.3029495e-03 -7.1168090e-03  6.4588725e-03  8.9729885e-03
 -5.0154282e-03 -3.7633716e-03  7.3805046e-03 -1.5334714e-03
 -4.5366134e-03  6.5540518e-03 -4.8601604e-03 -1.8160177e-03
  2.8765798e-03  9.9187379e-04 -8.2852151e-03 -9.4488179e-03
  7.3117660e-03  5.0702621e-03  6.7576934e-03  7.6286553e-04
  6.3508903e-03 -3.4053659e-03 -9.4640139e-04  5.7685734e-03
 -7.5216377e-03 -3.9361035e-03 -7.5115822e-03 -9.3004224e-04
  9.5381187e-03 -7.3191668e-03 -2.3337686e-03 -1.9377411e-03
  8.0774371e-03 -5.9308959e-03  4.5162440e-05 -4.7537340e-03
 -9.6035507e-03  5.0072931e-03 -8.7595852e-03 -4.3918253e-03
 -3.5099984e-05 -2.9618145e-04 -7.6612402e-03  9.6147433e-03
  4.9820580e-03  9.2331432e-03 -8.1579173e-03  4.4957981e-03
 -4.1370760e-03  8.2453608e-04  8.4986202e-03 -4.4621765e-03
  4.5175003e-03 -6.7869602e-03 -3.5484887e-03  9.3985079e-03
 -1.5776526e-03  3.2137157e-04 -4.1406299e-03 -7.6826881e-03
 -1.5080082e-03  2.46979

In [ ]:
import gensim.downloader as api

glove_vectors = api.load("glove-wiki-gigaword-100")  # Example: 100-dimensional GloVe

# Get word vectors (embeddings)
word1 = "king"
word2 = "queen"
vector1 = glove_vectors[word1]
vector2 = glove_vectors[word2]

# Compute cosine similarity between the two word vectors
similarity = glove_vectors.similarity(word1, word2)

print(f"Word vectors for '{word1}': {vector1}")
print(f"Word vectors for '{word2}': {vector2}")
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")


[--------------------------------------------------] 1.8% 2.4/128.1MB downloaded